In [1]:
# Imports
import astropy.units as u
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import scipy
from datetime import date
import time

matplotlib.use("tkagg")

In [2]:
# Define constants
sag_a_r_s = 1.2 * 10 ** 10 * u.m  # https://academic.oup.com/mnras/article/421/2/1315/1134797?login=false
r_g = sag_a_r_s / 2
c = 2.99 * 10 ** 8 * (u.m / u.s)

In [3]:
def q_squared_to_b_squared(q, v, r_s):
    """Convert from periastron to impact parameter b"""
    gamma = 1 / np.sqrt(1 - (v / c) ** 2)
    b_squared = q ** 2 * (c / v) ** 2 * (1 / (1 - r_s / q) - 1 / gamma ** 2)
    return b_squared

In [ ]:
def process_time_step_orbit_solve_ivp(t, r_from_q, energy_term, h, q, lightspeed, r_s):
    """Calculate changes to r and the vaporization loss, modified for solve_ivp"""
    r = abs(r_from_q) + q
    dr = -1 * np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))
    return dr + 0.01 


def process_radius_step_orbit_solve_ivp(r, t, energy_term, h, lightspeed, r_s):
    """Calculate the change in time for """
    d_tau = 1 / np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))
    d_t = 1 / (lightspeed * (1 - r_s / r) * np.sqrt(1 - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2) * 1 / energy_term))
    return np.array([d_tau, d_t])


def calc_radius_loss(distance, fudge, dt):
    dr_frag = fudge * 10 * u.m / (200 * u.s) * (distance / (10 * r_g)) ** -3 * dt
    return dr_frag


def check_if_correct_q(q, b, v):
    q = q * r_g
    total = ((b / r_g) ** 2 - q ** 2 * c ** 2 / v ** 2 * (1 / (1 - sag_a_r_s / q) - (1 - (v / c) ** 2)) / r_g ** 2).value
    return total

def numerically_determine_q(b, v):
    """Numerically determine the closest approach"""
    result = scipy.optimize.fsolve(check_if_correct_q, np.array([8]), args=(b, v))
    return result.x

In [10]:
# Integrate to find the time of periastron
def eqn_to_integrate_for_peri_time(r, energy_term, h, q, lightspeed, r_s):
    """Equation that gets integrated"""
    denom = np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))
    return -1 / denom


def calc_peri_time(q, args):
    """Integrate with quadrature to analytically find the periastron time"""
    result = scipy.integrate.quad(eqn_to_integrate_for_peri_time, 60 * r_g.value, q.value, args=args)
    return result

In [ ]:
# Integrate over an orbit using solve_ivp and return its luminosity curve
def integrate_luminosity(times, v_inf, b_inf, progenitor_radius, vaporization_fudge_factor, boulder_size_power_law, perihelion_time, min_boulder_size, max_boulder_size, prints=False):

    # Other parameters
    if prints:
        print(v_inf)
    initial_distance = 60 * r_g
    initial_radius = np.linspace(min_boulder_size, max_boulder_size, 1000) * u.m
    rho = 100 * u.kg / (u.m ** 3)
    epsilon = 0.1
    energy = c ** 2 * 1 / (1 - (v_inf / c) ** 2)
    angular_momentum_h = v_inf * b_inf * 1 / np.sqrt(1 - (v_inf / c) ** 2)

    solve_ivp_args = (energy.value, angular_momentum_h.value, c.value, sag_a_r_s.value)
    numerical_q = numerically_determine_q(b_inf, v_inf)
    start = time.perf_counter()
    ode_result = scipy.integrate.solve_ivp(process_radius_step_orbit_solve_ivp, (0, np.max(times) + 1), [distance_from_q.value], t_eval=times, args=solve_ivp_args, rtol=10**-10, atol=3)
    end = time.perf_counter()
    if prints:
        print(end - start)
        print("int finished")

    radius_loss_at_each_t = calc_radius_loss(abs(ode_result.y[0, :]) * u.m + q, vaporization_fudge_factor, 1 * u.s)
    total_radius_lost = np.cumulative_sum(radius_loss_at_each_t)
    radiuses_at_each_time = np.transpose(np.tile(total_radius_lost, (len(initial_radius), 1)))
    radius_at_t = np.add(-1 * radiuses_at_each_time, initial_radius)
    radius_at_t[radius_at_t < 0] = 0
    luminosities = (epsilon * c.value ** 2) * 4 * np.pi * rho * radius_at_t ** 2 * np.transpose(np.tile(radius_loss_at_each_t, (len(initial_radius), 1)))
    scale_factors = initial_radius ** -boulder_size_power_law
    total_volume = (4/3) * np.pi * progenitor_radius ** 3
    unscaled_fragment_volume = np.sum(scale_factors * (4/3) * np.pi * initial_radius ** 3)
    scale_factors_volume_adjusted = scale_factors * total_volume / unscaled_fragment_volume
    scaled_luminosities = np.sum(luminosities * scale_factors_volume_adjusted[None, :], axis=1)
    scaled_luminosities[abs(ode_result.y[0, :]) * u.m + q > 30 * r_g] = 0
    return scaled_luminosities

In [119]:
# Testing the tolerances and the runtime
brightness_6 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0)
brightness_8 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0)
brightness_4 = integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0)
plt.figure()
plt.plot(brightness_4, label="absolute tolerance=10^-2")
plt.plot(brightness_6, label="absolute tolerance=10^-6")
plt.plot(brightness_8, label="absolute tolerance=10^-4")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Luminosity")
plt.show()

(2233.041838408241, 6.148293323349208e-07) 0.0008024000562727451
int started
0.06573089997982606
int finished
(2233.041838408241, 6.148293323349208e-07) 0.0004708000342361629
int started
0.043741999950725585
int finished
(2233.041838408241, 6.148293323349208e-07) 0.000908799993339926
int started
0.13226360001135617
int finished


In [33]:
# Experiment with different variables
min_boulders_to_test = np.linspace(100, 500, 5)
plt.figure()
for s in min_boulders_to_test:
    plt.plot(integrate_luminosity(np.linspace(0, 4000, 4000), 180 * u.m / u.s, 5 * r_g, 5000 * u.m, 1, 1.8, 0, 10, 150), label=f"max_boulder={s}m")
    print(s)
plt.legend()
plt.xlabel("Time")
plt.ylabel("Luminosity")
plt.title(f"q=5r_g, v=180, r_iso=5000, slope=1.8, fudge=1, min_bor=10m")
# plt.ylim(bottom=0)
plt.show()

180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.0007878000033088028
int started
0.0406253999681212
int finished
100.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.00037759996484965086
int started
0.037635000015143305
int finished
200.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.0003931000246666372
int started
0.03614959999686107
int finished
300.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.0003934000269509852
int started
0.03360010002506897
int finished
400.0
180.0 m / s
(2233.041838408241, 6.148293323349208e-07) 0.00040499999886378646
int started
0.045041400007903576
int finished
500.0


In [13]:
def calc_error_in_flare_shape(inputs, flare_info, plot=False):
    """Calculate the error in flare shape for a given set of flare parameters"""

    # Define a constant to convert from JWST mJy to Watts
    fraction_of_luminosity_in_2_micron = 10 ** -27

    v_unitless, q_of_r_g, r_i, vap_fudge, boulder_size_power_law, perihelion_time, min_boulder_size, max_boulder_size, platform = inputs
    q = q_of_r_g * r_g
    v = v_unitless * u.m / u.s
    b_inf = np.sqrt(q_squared_to_b_squared(q, v, sag_a_r_s))
    energy = c ** 2 * 1 / (1 - (v / c) ** 2)
    angular_momentum_h = v * b_inf * 1 / np.sqrt(1 - (v / c) ** 2)
    args = (energy.value, angular_momentum_h.value, q.value, c.value, sag_a_r_s.value)
    peri_time = calc_peri_time(q, args)[0]
    obs_times = flare_info[:, 0]
    times = peri_time + (obs_times - perihelion_time)
    flare_brightness = integrate_luminosity(times, v, q, r_i, vap_fudge, boulder_size_power_law, perihelion_time, min_boulder_size, max_boulder_size, plot)
    scaled_brightness = flare_brightness * fraction_of_luminosity_in_2_micron
    if plot:
        plt.figure()
        plt.plot(times, flare_info[:, 1] - platform, label="Observed")
        plt.plot(times, scaled_brightness, label="Model")
        plt.legend()
        plt.xlabel("Time")
        plt.ylabel("Luminosity [mJy]")
        plt.show()
    error = flare_info[:, 1] - platform - scaled_brightness.value
    return np.sum(error ** 2)

In [ ]:
# Load in section of observed light curve
jwst_data = np.loadtxt('sgra_frame_lightcurve_all.csv', delimiter=',', skiprows=1, usecols=(4, 7, 8))
plt.figure()
plt.plot(jwst_data[:, 1])
plt.show()
section_around_flare = jwst_data[abs(jwst_data[:, 0] - 60405.35070) * 24 * 3600 < 400]
section_around_flare[:, 0] = (section_around_flare[:, 0] - 60405.35070) * 24 * 3600


In [63]:
guess = np.array([10 ** 5, 4.1, 3700, 1, 2, 420, 1, 200, 1.3])
calc_error_in_flare_shape(guess, section_around_flare, plot=True)

100000.0 m / s
(5571.462989673468, 3.035871577594662e-05) 0.0005385000004025642
int started
0.015652599999157246
int finished


np.float64(33.41778744235849)

In [64]:
# Minimize the error to find the optimal orbit
result = scipy.optimize.minimize(calc_error_in_flare_shape, guess, bounds=((1, 10 ** 6), (4.1, 30), (1, 1000000), (0.01, 10), (0.1, 10), (-1000, 1000), (1, 10), (100, 300), (-10, 10)), args=section_around_flare)
calc_error_in_flare_shape(result.x, section_around_flare, True)
print(result.x, calc_error_in_flare_shape(result.x, section_around_flare))

Exception ignored in: <function WeakSet.__init__.<locals>._remove at 0x0000010A2425D580>
Traceback (most recent call last):
  File "C:\Users\thkel\AppData\Local\Programs\Python\Python313\Lib\_weakrefset.py", line 39, in _remove
    def _remove(item, selfref=ref(self)):
KeyboardInterrupt: 

KeyboardInterrupt



In [65]:
result = scipy.optimize.differential_evolution(calc_error_in_flare_shape, ((1, 10 ** 6), (4.1, 30), (1, 1000000), (0.01, 10), (0.1, 10), (-1000, 1000), (1, 10), (100, 300), (-10, 10)), args=(section_around_flare,))
calc_error_in_flare_shape(result.x, section_around_flare, True)
print(result.x, calc_error_in_flare_shape(result.x, section_around_flare))

C:\Users\thkel\AppData\Local\Temp\ipykernel_23584\313026193.py:14: RuntimeWarning: invalid value encountered in sqrt
  dr = -1 * np.sqrt(energy_term - (1 - r_s / r) * (lightspeed ** 2 + h ** 2 / r ** 2))


601744.5981527577 m / s
(6391.143962635566, 3.041895979549736e-08) 0.0007501000000047497
int started
0.01570259999971313
int finished
[6.01744598e+05 2.99346676e+01 7.53617979e+05 2.88727742e+00
 7.93905204e+00 8.61701875e+02 4.65908555e+00 2.76686722e+02
 1.54105074e+00] 18.33358577897863
